In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from joblib import Parallel, delayed

from gower_similarity.core.similarity import GowerSimilarity

In [ ]:
# load and preprocess data
n_rows = 1000
df = pd.read_csv('<your_path>/adult.csv').head(n_rows)

feature_cols = ['age', 'educational-num', 'race', 'gender', 'hours-per-week', 'relationship', 'occupation', 'education', 'workclass']
target_col = 'income'

# convert income to 0 if <=50K, 1 if >50K
X = df[feature_cols]
y = df[target_col].apply(lambda x: 1 if x == '>50K' else 0)

In [ ]:
# process data -> where spot '?' replace with NaN
X.replace(to_replace='?', value=np.nan, inplace=True)

In [ ]:
# define features types
feature_types = {
    "age": "ratio_scale_interval",
    "educational-num": "ratio_scale_interval",
    "race": "categorical_nominal",
    "gender": "categorical_nominal",
    "hours-per-week": "ratio_scale_interval",
    "relationship": "categorical_nominal",
    "occupation": "categorical_nominal",
    "education": "categorical_nominal",
    "workclass": "categorical_nominal"
}

In [ ]:
# fit Gower instance
gs = GowerSimilarity(feature_types=feature_types).fit(X)

In [ ]:
# split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# now we have to create custom distance matrices to pass them into kNN classifier
# first one should be with shape (n_train, n_train), the second one with shape (n_test, n_train)
# we will use 'precomputed' metric for kNN classifier
train_matrix = np.zeros((X_train.shape[0], X_train.shape[0]), dtype=np.float32)
test_matrix = np.zeros((X_test.shape[0], X_train.shape[0]), dtype=np.float32)

In [ ]:
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()

In [ ]:
def compute_row_train(i):
    xi = X_train[i]
    return [gs.distance(xi, xj) for xj in X_train]

def compute_row_test(i):
    xi = X_test[i]
    return [gs.distance(xi, xj) for xj in X_train]

In [ ]:
# fill matrices using GowerSimilarity instance
train_matrix = np.array(Parallel(n_jobs=-1, verbose=1)(delayed(compute_row_train)(i) for i in range(len(X_train)))).astype(np.float32)
test_matrix = np.array(Parallel(n_jobs=-1, verbose=1)(delayed(compute_row_test)(i) for i in range(len(X_test)))).astype(np.float32)

In [ ]:
# now we can create kNN classifier
knn = KNeighborsClassifier(n_neighbors=5, metric='precomputed')

In [ ]:
# fit the model
knn.fit(train_matrix, y_train)

In [ ]:
# predict on the test data
y_pred = knn.predict(test_matrix)

In [ ]:
# evaluate the classifier
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred))